# 03b · Retrieval benchmark: PlantIQ ablation + Selva Kumar et al. retriever

Every system gets the same question **and the same disease label** (as the CNN diagnosis would provide).

| Config | What is retrieved |
|---|---|
| Dense only | bge-m3 FAISS top-k on PlantIQ's expanded query |
| BM25 only | BM25 top-k on the same query |
| Dense+BM25 (RRF) | reciprocal-rank fusion of the two |
| RRF + rerank | RRF pool of 20, reordered by the bge-reranker cross-encoder |
| Full PlantIQ | the app's `search()`: RRF → topic filter → rerank → MMR |
| Selva Kumar et al. | LangChain RecursiveCharacterTextSplitter (1000/200) → embeddings → FAISS similarity top-k |

Metrics per question (in-KB questions only): Recall@k (share of gold quotes found in the top k), Hit@k, nDCG@k,
MRR@10; reported overall and for English / Kannada. **No API key needed**: PlantIQ's language router runs
in its built-in offline mode, so Kannada questions are searched as written (bge-m3 embeds Kannada natively).
Time: ~10 min (the first run downloads ~4.5 GB of models).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "03_rag"))

import json
import os

import numpy as np
import pandas as pd

from common import config, env
from common import rag_eval as R
from plantiq_rag_snapshot.advisory import display_name, retrieve
from plantiq_rag_snapshot.chunking import DISEASE_TOPIC
from plantiq_rag_snapshot.build import build_retriever
from plantiq_rag_snapshot.profiles import BALANCED, QUALITY
from plantiq_rag_snapshot.retriever import rrf_fuse

config.ensure_dirs()
device = env.require_cuda().type
qs = pd.read_json(config.RAG_DATA / "questions.jsonl", lines=True)
qs = qs[qs.out_of_kb == 0].reset_index(drop=True)
print(len(qs), "in-KB questions")

PROFILE = BALANCED if config.SMOKE else QUALITY
SK_EMB = "hf:sentence-transformers/all-MiniLM-L6-v2" if config.SMOKE else \
    os.getenv("SK_EMBEDDINGS", "openai:text-embedding-3-small" if os.getenv("OPENAI_API_KEY", "").strip() else
              "hf:sentence-transformers/all-mpnet-base-v2")
print("PlantIQ profile:", PROFILE.name, "| SK embeddings:", SK_EMB)

In [ ]:
rag = build_retriever(PROFILE, config.KB_DIR, config.RAG_INDEX / f"pq_{PROFILE.name}", device=device)
sk = R.SKRetriever(SK_EMB, config.KB_DIR, config.RAG_INDEX, device=device)
llm = None   # PlantIQ's router falls back to its offline heuristic (no API calls)
print(rag.info(), "| SK chunks:", len(sk.chunks))

In [ ]:
KS = (1, 3, 5, 10)

def ranked_texts(ids):
    return [rag.chunks[i]["text"] for i in ids]

rows = []
for _, q in qs.iterrows():
    quotes = list(q["evidence"])
    _, info, _ = retrieve(disease=q["disease"], question=q["question"], rag=rag, llm=llm, language=q["lang"])
    query, rq = info["query"], info["rerank_query"]
    dense, sparse = rag._dense(query, 20), rag._sparse(query, 20)
    fused = rrf_fuse([dense, sparse])
    rrf_order = sorted(fused, key=fused.get, reverse=True)
    reranked = rag._rerank(rq, [(i, rag.chunks[i]) for i in rrf_order[:20]], fused)
    lists = {
        "Dense only": ranked_texts(sorted(dense, key=dense.get)[:10]),
        "BM25 only": ranked_texts(sorted(sparse, key=sparse.get)[:10]),
        "Dense+BM25 (RRF)": ranked_texts(rrf_order[:10]),
        "RRF + rerank": [r["text"] for r in reranked[:10]],
    }
    full = {k: [r["text"] for r in rag.search(query, top_k=k, topic_filter=DISEASE_TOPIC.get(q["disease"]),
                                               rerank_query=rq)] for k in (3, 5, 10)}
    sk_query = f"{display_name(q['disease'])}: {q['question']}"
    sk_list = [h["text"] for h in sk.search(sk_query, k=10)]

    def score(name, ranked):
        m = R.retrieval_metrics(R.chunk_relevance(ranked, quotes), ks=KS)
        rows.append({"id": q["id"], "topic": q["topic"], "lang": q["lang"], "system": name, **m})

    for name, ranked in lists.items():
        score(name, ranked)
    # Full PlantIQ: the app returns top_k=3; @1/@3 come from that call, @5 and @10/MRR from top_k=5/10 calls
    m3 = R.retrieval_metrics(R.chunk_relevance(full[3], quotes), ks=(1, 3))
    m5 = R.retrieval_metrics(R.chunk_relevance(full[5], quotes), ks=(5,))
    m10 = R.retrieval_metrics(R.chunk_relevance(full[10], quotes), ks=(10,))
    rows.append({"id": q["id"], "topic": q["topic"], "lang": q["lang"], "system": "Full PlantIQ (+topic, MMR)",
                 **{k: v for k, v in m3.items() if k != "mrr@10"}, **{k: v for k, v in m5.items() if "5" in k},
                 **{k: v for k, v in m10.items() if "10" in k}})
    score("Selva Kumar et al. (dense, LangChain)", sk_list)

ret = pd.DataFrame(rows)
ret.to_csv(config.RAG_RESULTS / "retrieval_rows.csv", index=False)
ORDER = ["Selva Kumar et al. (dense, LangChain)", "Dense only", "BM25 only", "Dense+BM25 (RRF)", "RRF + rerank",
         "Full PlantIQ (+topic, MMR)"]
cols = ["recall@1", "recall@3", "recall@5", "recall@10", "hit@3", "ndcg@3", "mrr@10"]
summary = ret.groupby("system")[cols].mean().reindex(ORDER)
summary

In [ ]:
by_lang = ret.groupby(["system", "lang"])[["recall@3", "ndcg@3"]].mean().unstack().reindex(ORDER)
by_lang.columns = [f"{m} ({l})" for m, l in by_lang.columns]
ci = {s: {m: R.bootstrap_mean_ci(ret.loc[ret.system == s, m]) for m in ("recall@3", "ndcg@3")} for s in ORDER}
pq = ret[ret.system == "Full PlantIQ (+topic, MMR)"].set_index("id")
tests = {s: {m: R.wilcoxon_paired(pq[m], ret[ret.system == s].set_index("id").loc[pq.index, m])
             for m in ("recall@3", "ndcg@3")} for s in ORDER if s != "Full PlantIQ (+topic, MMR)"}
env.write_json(config.RAG_RESULTS / "retrieval.json", {
    "n_questions": int(len(qs)), "profile": PROFILE.name, "sk_embeddings": SK_EMB,
    "plantiq_chunks": len(rag.chunks), "sk_chunks": len(sk.chunks), "match_threshold": R.MATCH_THRESHOLD,
    "summary": summary.to_dict(orient="index"), "by_lang": by_lang.to_dict(orient="index") if len(by_lang) else {},
    "ci": ci, "wilcoxon_full_vs": tests, "meta": env.run_meta()})
by_lang